# Automated Stock Data Downloader

This notebook automates the download of stock price data from Yahoo Finance:

1. **Auto-detect last download date** - Finds the most recent CSV file to determine start date
2. **Download new data** - Fetches 5-minute interval data from last download to today (or custom date range)
3. **Save with consistent naming** - Uses the `newf_raw_prices_{start}_{end}.csv` convention
4. **Log all operations** - Creates a detailed log file with download statistics

In [1]:
import glob
import os
import re
from datetime import datetime, timedelta
import pandas as pd
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Stock list - same as original notebook
stocks = 'NDAQ GOOG IBM NFLX SAM AMZN AAPL NVDA BMY BAYRY APPN SGMO FTNT MSFT PFE NVS NICE SWKS PSTG PATH ASML ZM SNOW AVNT ACN JNJ PFE ABBV MRK AMGN SNY BMY GILD LLY GSK AZN BIIB REGN VRTX ZTS SGMO TEVA COST PWR TTD AES SCHD VOO'.split()
stocks = list(set(stocks))  # Remove duplicates

print(f"Tracking {len(stocks)} unique stocks")
print(f"Stocks: {', '.join(sorted(stocks))}")

Tracking 45 unique stocks
Stocks: AAPL, ABBV, ACN, AES, AMGN, AMZN, APPN, ASML, AVNT, AZN, BAYRY, BIIB, BMY, COST, FTNT, GILD, GOOG, GSK, IBM, JNJ, LLY, MRK, MSFT, NDAQ, NFLX, NICE, NVDA, NVS, PATH, PFE, PSTG, PWR, REGN, SAM, SCHD, SGMO, SNOW, SNY, SWKS, TEVA, TTD, VOO, VRTX, ZM, ZTS


In [ ]:
def format_filename_date(dt):
    """Format a datetime as a simple date string for use in filenames. e.g. 2026-02-03"""
    return pd.Timestamp(dt).strftime('%Y-%m-%d')


def parse_filename_date(date_str):
    """
    Parse a filename date string back to a pandas Timestamp.
    Handles: simple date (2026-02-03), Windows-safe (2026-02-03T20-55-00UTC),
    and legacy Mac format (2026-02-03 20:55:00+00:00).
    """
    # Simple date: 2026-02-03
    if re.match(r'^\d{4}-\d{2}-\d{2}$', date_str):
        return pd.to_datetime(date_str)

    # Windows-safe timestamp: 2026-02-03T20-55-00UTC
    new_fmt = re.match(r'(\d{4}-\d{2}-\d{2})T(\d{2})-(\d{2})-(\d{2})UTC', date_str)
    if new_fmt:
        normalised = f"{new_fmt.group(1)} {new_fmt.group(2)}:{new_fmt.group(3)}:{new_fmt.group(4)}+00:00"
        return pd.to_datetime(normalised, utc=True)

    # Legacy Mac format: spaces and colons
    try:
        return pd.to_datetime(date_str, utc=True)
    except Exception:
        return None


def find_latest_download_date(data_dir="./stockcharts_data/data_yfinance/"):
    """
    Find the most recent end date from existing CSV files.
    Handles simple date format, Windows-safe timestamp format, and legacy Mac filenames.

    Returns:
        datetime: The latest date found in CSV filenames, or None if no files found
    """
    csv_pattern = os.path.join(data_dir, "newf_raw_prices*.csv")
    csv_files = glob.glob(csv_pattern)

    if not csv_files:
        print("No existing CSV files found. Will start from default date.")
        return None

    latest_date = None
    latest_file = None

    # Matches:
    #   simple:  newf_raw_prices_2026-02-01_2026-02-03.csv
    #   legacy:  newf_raw_prices_2026-02-01 14:30:00+00:00_2026-02-03 20:55:00+00:00.csv
    date_pattern = r'newf_raw_prices_(.+)_([^_]+)\.csv$'

    for file in csv_files:
        filename = os.path.basename(file)
        match = re.search(date_pattern, filename)

        if match:
            end_date_str = match.group(2)
            end_date = parse_filename_date(end_date_str)
            if end_date is not None:
                if latest_date is None or end_date > latest_date:
                    latest_date = end_date
                    latest_file = filename
            else:
                print(f"Warning: Could not parse date from {filename}")

    if latest_date:
        print(f"Latest download found: {latest_file}")
        print(f"Latest end date: {latest_date.date() if hasattr(latest_date, 'date') else latest_date}")

    return latest_date


def download_stock_data(stocks, start_date, end_date, interval="5m"):
    """
    Download stock data using yfinance.

    Args:
        stocks: List of stock tickers
        start_date: Start date for download (datetime or string)
        end_date: End date for download (datetime or string)
        interval: Data interval (default: "5m")

    Returns:
        DataFrame: Close prices for all stocks
    """
    print(f"\nDownloading data from {start_date} to {end_date}...")
    print(f"Interval: {interval}")
    print(f"Stocks: {len(stocks)} tickers")

    try:
        data = yf.download(stocks, start=start_date, end=end_date, interval=interval)
        close_prices = data['Close']

        print(f"\nDownload complete!")
        print(f"Data shape: {close_prices.shape}")
        print(f"Rows (time periods): {len(close_prices)}")
        print(f"Columns (stocks): {len(close_prices.columns)}")

        return close_prices

    except Exception as e:
        print(f"Error downloading data: {e}")
        return None


def save_data_with_log(df, data_dir="./stockcharts_data/data_yfinance/", log_dir="./logs/"):
    """
    Save downloaded data to CSV and create a log file.

    Args:
        df: DataFrame to save
        data_dir: Directory to save CSV files
        log_dir: Directory to save log files

    Returns:
        dict: Summary of save operation
    """
    if df is None or df.empty:
        print("No data to save.")
        return None

    # Create directories if they don't exist
    os.makedirs(data_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)

    # Get date range from data
    min_date = df.index.min()
    max_date = df.index.max()

    # Simple date-only filename - works on Windows, Mac, and Linux
    csv_filename = f"newf_raw_prices_{format_filename_date(min_date)}_{format_filename_date(max_date)}.csv"
    csv_path = os.path.join(data_dir, csv_filename)

    # Save CSV
    df.to_csv(csv_path)
    print(f"\nData saved to: {csv_path}")

    # Calculate statistics
    file_size = os.path.getsize(csv_path)
    file_size_mb = file_size / (1024 * 1024)
    num_rows = len(df)
    num_cols = len(df.columns)
    num_data_points = num_rows * num_cols

    # Count non-null values
    non_null_count = df.notna().sum().sum()
    null_count = df.isna().sum().sum()

    # Create log entry
    log_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_filename = f"download_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    log_path = os.path.join(log_dir, log_filename)

    log_content = f"""Stock Data Download Log
{'=' * 60}
Download Timestamp: {log_timestamp}
Data Date Range: {format_filename_date(min_date)} to {format_filename_date(max_date)}

FILE INFORMATION:
  Filename: {csv_filename}
  Full Path: {csv_path}
  File Size: {file_size_mb:.2f} MB ({file_size:,} bytes)

DATA STATISTICS:
  Time Periods (Rows): {num_rows:,}
  Stock Tickers (Columns): {num_cols}
  Total Data Points: {num_data_points:,}
  Valid Data Points: {non_null_count:,}
  Missing Data Points: {null_count:,}
  Data Completeness: {(non_null_count/num_data_points*100):.2f}%

STOCK TICKERS:
  {', '.join(sorted(df.columns.tolist()))}

DATE RANGE DETAILS:
  First Timestamp: {df.index.min()}
  Last Timestamp: {df.index.max()}
  Duration: {(max_date - min_date).days} days
  Total Time Points: {len(df)}

PER-STOCK STATISTICS:
"""

    for col in sorted(df.columns):
        col_non_null = df[col].notna().sum()
        col_completeness = (col_non_null / len(df)) * 100
        log_content += f"  {col}: {col_non_null}/{len(df)} points ({col_completeness:.1f}%)\n"

    log_content += "\n" + "=" * 60 + "\n"

    with open(log_path, 'w') as f:
        f.write(log_content)

    print(f"Log saved to: {log_path}")

    master_log_path = os.path.join(log_dir, "master_download_log.txt")
    with open(master_log_path, 'a') as f:
        f.write(f"\n{log_timestamp} | {csv_filename} | {num_rows} rows | {num_cols} stocks | {file_size_mb:.2f} MB\n")

    print(f"Entry added to master log: {master_log_path}")

    return {
        'csv_path': csv_path,
        'log_path': log_path,
        'file_size_mb': file_size_mb,
        'num_rows': num_rows,
        'num_cols': num_cols,
        'date_range': (min_date, max_date)
    }

## Download New Data

This cell will:
1. Find the last download date automatically
2. Download data from that date to today
3. Save with proper naming convention
4. Generate detailed logs

### To customize dates:
Set `custom_start_date` and/or `custom_end_date` to override automatic detection.

Example:
```python
custom_start_date = "2026-01-21"  # Override start date
custom_end_date = "2026-02-10"    # Override end date
```

In [4]:
# ============================================
# CONFIGURATION - Customize these if needed
# ============================================

# Set to None to auto-detect, or specify a date string like "2026-01-21"
custom_start_date = "2026-02-01"
custom_end_date = None  # None means use today's date

# Data directory
data_dir = "./stockcharts_data/data_yfinance/"
log_dir = "./logs/"

# ============================================
# MAIN EXECUTION
# ============================================

print("="*60)
print("STOCK DATA DOWNLOADER")
print("="*60)

# Determine start date
if custom_start_date:
    start_date = pd.to_datetime(custom_start_date)
    print(f"\nUsing custom start date: {start_date}")
else:
    latest_date = find_latest_download_date(data_dir)
    if latest_date:
        # Start from the day after the latest download
        start_date = latest_date + timedelta(days=1)
        print(f"\nAuto-detected start date: {start_date}")
    else:
        # Default to 30 days ago if no files found
        start_date = datetime.now() - timedelta(days=30)
        print(f"\nNo existing files found. Using default start date: {start_date}")

# Determine end date
if custom_end_date:
    end_date = pd.to_datetime(custom_end_date)
    print(f"Using custom end date: {end_date}")
else:
    end_date = datetime.now()
    print(f"Using today as end date: {end_date}")

# Download data
downloaded_data = download_stock_data(stocks, start_date, end_date, interval="5m")

# Save data and create logs
if downloaded_data is not None and not downloaded_data.empty:
    summary = save_data_with_log(downloaded_data, data_dir, log_dir)
    
    if summary:
        print("\n" + "="*60)
        print("DOWNLOAD SUMMARY")
        print("="*60)
        print(f"CSV File: {summary['csv_path']}")
        print(f"Log File: {summary['log_path']}")
        print(f"File Size: {summary['file_size_mb']:.2f} MB")
        print(f"Data Points: {summary['num_rows']:,} rows x {summary['num_cols']} stocks")
        print(f"Date Range: {summary['date_range'][0]} to {summary['date_range'][1]}")
        print("="*60)
else:
    print("\nNo data was downloaded. Check date range and stock symbols.")

STOCK DATA DOWNLOADER

Using custom start date: 2026-02-01 00:00:00
Using today as end date: 2026-02-04 15:31:31.472713

Interval: 5m
Stocks: 45 tickers
YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  45 of 45 completed


Download complete!
Data shape: (180, 45)
Rows (time periods): 180
Columns (stocks): 45

Data saved to: ./stockcharts_data/data_yfinance/newf_raw_prices_2026-02-02 18:30:00+00:00_2026-02-04 20:30:00+00:00.csv
Log saved to: ./logs/download_log_20260204_153132.txt
Entry added to master log: ./logs/master_download_log.txt

DOWNLOAD SUMMARY
CSV File: ./stockcharts_data/data_yfinance/newf_raw_prices_2026-02-02 18:30:00+00:00_2026-02-04 20:30:00+00:00.csv
Log File: ./logs/download_log_20260204_153132.txt
File Size: 0.14 MB
Data Points: 180 rows x 45 stocks
Date Range: 2026-02-02 18:30:00+00:00 to 2026-02-04 20:30:00+00:00


## View Recent Logs

Run this cell to view the master log of all downloads.

In [5]:
# View master log
master_log_path = "./logs/master_download_log.txt"

if os.path.exists(master_log_path):
    print("MASTER DOWNLOAD LOG")
    print("="*80)
    with open(master_log_path, 'r') as f:
        print(f.read())
else:
    print("No master log file found yet. Run a download first.")

MASTER DOWNLOAD LOG

2026-02-04 15:31:32 | newf_raw_prices_2026-02-02 18:30:00+00:00_2026-02-04 20:30:00+00:00.csv | 180 rows | 45 stocks | 0.14 MB

